# Foodie Advisor · 3. Costos, scorecard, validación temporal y producto

Este notebook reúne las fases `07_costos_multiciudad`, `08_scorecard`, `09_validacion_temporal`. Cada sección escribe su `resultados/hallazgos_XX.json`, que es lo que lee la app. Al final, la sección **El producto** explica cómo correr la app del gerente y arma su ficha para C_Tapas.

## 07 · Cuánto cuesta cada falla, en 6 ciudades, y qué afirmaciones del avance sobreviven

| | |
|---|---|
| **Qué entra** | reseñas de las 6 ciudades (00); más adelante, las predicciones del clasificador elegido en el 06 |
| **Qué sale** | `data/procesado/costos_*.csv`, forest plot por etapa y ciudad (`figuras/07_*.png`) |

**Estado:** final. Titular de P1: costo de una queja explícita (regex v1). LogReg, elegido por la regla de selección pero no apto (06), entra solo como análisis de sensibilidad.

In [1]:
import sys, time
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import spearmanr
from foodie_advisor import CIUDADES, FIGURAS, PROCESADO, datos, lexico, costos, etiquetas as et, modelos as m, resultados
from foodie_advisor import scorecard as sc

sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams["font.family"] = "DejaVu Sans"
AZUL, ROJO, GRIS = "#3b6ea5", "#b5483a", "#8c8c8c"
COLOR_CIUDAD = dict(zip(CIUDADES, ["#b5483a", "#3b6ea5", "#5b8c5a", "#e3a33b", "#7a5195", "#8c8c8c"]))
NOMBRE = {e: et.NOMBRE[e] for e in et.ETIQUETAS}

### 1. Línea base: costo por etapa con el regex del avance, en las 6 ciudades

El modelo es el del avance: la regresión de las estrellas sobre 7 indicadores por reseña (¿menciona una falla de esta etapa?), con efectos fijos por restaurante. El coeficiente compara reseñas **del mismo local**. Los IC 95% salen de 200 remuestreos de restaurantes completos.

Esta es la cifra contra la que se compara el clasificador. Como el regex solo ve quejas explícitas, y las explícitas suelen ser las más duras, lo esperable es que **sobreestime** el costo.

In [2]:
ruta = PROCESADO / "costos_regex_v1.csv"
if ruta.exists():
    base = pd.read_csv(ruta, index_col=[0, 1])
else:
    partes = {}
    for c in CIUDADES:
        t0 = time.time()
        r = datos.cargar_resenas(c, ["restaurante", "estrellas", "texto"])
        Y = et.etiquetas_regex(r.texto, lexico.ETAPAS)
        partes[c] = costos.costos_ciudad(r, Y, reps=200)
        print(f"{c:10} {len(r):>9,} reseñas  {time.time() - t0:5.0f} s")
    base = pd.concat(partes, names=["ciudad", "etapa"])
    base.to_csv(ruta)

mad = base.loc["Madrid", "β"].round(2)
avance = {"atencion": -1.58, "cobro": -0.98, "cocina_mesa": -0.75, "llegada": -0.20}
assert all(mad[e] == v for e, v in avance.items()), mad
print("Madrid reproduce el avance:", {e: mad[e] for e in avance})
base["β"].unstack("ciudad")[CIUDADES].loc[et.ETIQUETAS].round(2)

Madrid reproduce el avance: {'atencion': -1.58, 'cobro': -0.98, 'cocina_mesa': -0.75, 'llegada': -0.2}


ciudad,Madrid,Barcelona,London,Paris,New_York,New_Delhi
etapa,,,,,,
llegada,-0.20,-0.08,-0.22,-0.18,-0.13,-0.15
atencion,-1.58,-1.57,-1.48,-1.56,-1.39,-1.34
cocina_mesa,-0.75,-0.74,-0.86,-0.65,-0.76,-0.60
cobro,-0.98,-1.05,-0.80,-1.02,-1.02,-0.93
sabor,-1.18,-1.21,-1.22,-1.25,-1.12,-0.92
precio,-0.62,-0.61,-0.50,-0.45,-0.49,-0.37
local,-0.39,-0.51,-0.42,-0.49,-0.33,-0.48


In [3]:
def forest(tabla, titulo, archivo, ciudades=CIUDADES[:-1]):
    fig, ax = plt.subplots(figsize=(8, 5.2))
    etapas = et.ETIQUETAS[::-1]
    alto = 0.8 / len(ciudades)
    for j, c in enumerate(ciudades):
        sub = tabla.loc[c].loc[etapas]
        y = np.arange(len(etapas)) + (j - (len(ciudades) - 1) / 2) * alto
        ax.errorbar(sub["β"], y, xerr=[sub["β"] - sub["IC inf"], sub["IC sup"] - sub["β"]], fmt="o", ms=4,
                    color=COLOR_CIUDAD[c], label=c, capsize=2, lw=1)
    ax.axvline(0, color=GRIS, lw=1)
    ax.set_yticks(np.arange(len(etapas)))
    ax.set_yticklabels([NOMBRE[e] for e in etapas])
    for k in range(len(etapas) - 4, len(etapas)):
        ax.get_yticklabels()[k].set_color(ROJO)
    ax.set(xlabel="Cambio en estrellas al mencionar la falla (mismo restaurante)", title=titulo)
    ax.legend(fontsize=8, loc="lower left")
    plt.tight_layout(); plt.savefig(FIGURAS / archivo, dpi=150); plt.show()

forest(base, "Costo de cada falla por ciudad (regex v1; rojo: ciclo de servicio)", "07_forest_regex_v1.png")

<Figure size 800x520 with 1 Axes>

Con el regex del avance, el orden de Madrid se repite en las 6 ciudades. **La atención en sala es la falla más cara en todas**: va de −1.39★ (New York) a −1.58★ (Madrid), y en New Delhi es −1.34★. Detrás vienen el cobro (de −0.80★ en London a −1.05★ en Barcelona) y el paso de cocina a mesa (de −0.65★ en Paris a −0.86★ en London). La llegada cuesta poco en todas partes (de −0.08★ a −0.22★), lo que cuadra con el 03: la mayoría de sus menciones no son fallas. Fuera del ciclo, el sabor (de −1.12★ a −1.25★) cuesta casi tanto como la atención.

New Delhi va al apéndice: sigue el mismo orden, pero con costos más chicos en casi todo (sabor −0.92★) y con menos reseñas.

### 2. ¿El texto trae la calificación, o la trae el autor?

En el avance, una regresión Ridge sobre TF-IDF predecía las estrellas en restaurantes no vistos con un R² de 0.68. Pero entre el 58% y el 69% de las reseñas viene de autores que escriben más de una (00). Si un autor aparece en train y en test, el modelo puede estar reconociendo **su estilo** (alguien que siempre pone 5★) más que lo que dice el texto. Lo medimos sacando del test a los autores que también están en train.

In [4]:
r = datos.cargar_resenas("Madrid", ["restaurante", "estrellas", "texto", "autor"])
r2 = pd.DataFrame({"Partición por restaurante (avance)": m.r2_estrellas(r),
                   "Por restaurante y sin autores de train": m.r2_estrellas(r, excluir_autores=True)}).T
r2.round(3)

,reseñas test,R²,MAE,MAE media
Partición por restaurante (avance),40335.0,0.681,0.438,0.840
Por restaurante y sin autores de train,20417.0,0.708,0.420,0.883


La fuga por autor no infla el resultado. Partiendo solo por restaurante, se reproduce el R² del avance (0.681). Al sacar del test a todos los autores que también escriben en train, el R² no baja: sube a 0.708, sobre 20,417 reseñas. El texto predice la calificación por lo que dice, no porque el modelo reconozca a quien escribe. La cifra del avance se mantiene y se puede reportar con las dos particiones.

### 3. P2-P4, y el costo con el modelo como análisis de sensibilidad

El 06 concluyó que **ningún modelo aprendido es apto** para un gerente, así que el producto usa el regex v1, que sí cumple la precisión. Por eso **el titular de P1 es el costo de una queja explícita** (sección 1: β del regex v1, con IC bootstrap y réplica en 6 ciudades; es asociación, no causa). LogReg, el modelo que eligió la regla de selección, entra aquí solo como **análisis de sensibilidad**. Su error es diferencial (se dispara con el tono negativo en general), así que su β no se puede leer como un límite del β real. P2-P4 se calculan con las dos fuentes.

In [5]:
h06 = resultados.cargar("06")
FINAL, VARIANTE = h06["final"], h06["variante"]
FUENTE_M = sc.predecir_modelo("Madrid", FINAL, VARIANTE)          # LogReg (sensibilidad)
for c in CIUDADES:
    sc.predecir_modelo(c, FINAL, VARIANTE); sc.predecir_regex(c)
FUENTES = {"regex v1 (titular)": "regex_v1", f"{FINAL} (sensibilidad)": FUENTE_M}
print(f"Producto: {h06['producto']} | selección: {FINAL} ({VARIANTE}) | fuentes: {FUENTES}")

def por_resena(c, fuente):
    pred = pd.read_parquet(sc.ruta_pred(fuente, c))
    r = datos.cargar_resenas(c, ["review_id", "restaurante", "estrellas", "fecha"])
    return r.join(pred.groupby("review_id")[et.ETIQUETAS].max(), on="review_id").fillna({e: 0 for e in et.ETIQUETAS})

Producto: Regex v1 | selección: LogReg (borrado) | fuentes: {'regex v1 (titular)': 'regex_v1', 'LogReg (sensibilidad)': 'LogReg_borrado'}


#### 3.1 P1 · Análisis de sensibilidad: el costo con LogReg

In [6]:
ruta_m = PROCESADO / f"costos_{FUENTE_M}.csv"
if ruta_m.exists():
    con_modelo = pd.read_csv(ruta_m, index_col=[0, 1])
else:
    partes = {}
    for c in CIUDADES:
        r = por_resena(c, FUENTE_M)
        partes[c] = costos.costos_ciudad(r, r[et.ETIQUETAS].astype(float), reps=200)
    con_modelo = pd.concat(partes, names=["ciudad", "etapa"]); con_modelo.to_csv(ruta_m)

comp = pd.DataFrame({"β regex v1 (titular)": base["β"], "IC inf regex": base["IC inf"], "IC sup regex": base["IC sup"],
                     f"β {FINAL} (sensibilidad)": con_modelo["β"],
                     "% reseñas regex v1": base["% de reseñas"], f"% reseñas {FINAL}": con_modelo["% de reseñas"]})
display(comp.loc[(slice(None), et.CICLO), :].round(2))
forest(con_modelo, f"Sensibilidad: costo por falla con {FINAL} (rojo: ciclo de servicio)", "07_forest_sensibilidad.png")

,,β regex v1 (titular),IC inf regex,IC sup regex,β LogReg (sensibilidad),% reseñas regex v1,% reseñas LogReg
ciudad,etapa,,,,,,
Madrid,llegada,-0.20,-0.28,-0.14,-0.05,0.89,8.80
Barcelona,llegada,-0.08,-0.14,-0.03,-0.02,1.84,11.35
London,llegada,-0.22,-0.28,-0.18,-0.11,1.83,11.83
Paris,llegada,-0.18,-0.28,-0.08,-0.07,1.39,10.66
New_York,llegada,-0.13,-0.17,-0.09,-0.03,1.51,13.97
New_Delhi,llegada,-0.15,-0.32,-0.03,-0.04,0.96,7.42
Madrid,atencion,-1.58,-1.63,-1.53,-0.65,2.49,22.07
Barcelona,atencion,-1.57,-1.60,-1.53,-0.69,2.85,23.66
London,atencion,-1.48,-1.51,-1.45,-0.54,2.50,26.26


<Figure size 800x520 with 1 Axes>

In [7]:
fig, ax = plt.subplots(figsize=(7.5, 3.6))
mad = comp.loc["Madrid"].loc[et.ETIQUETAS[::-1]]
y = np.arange(len(mad))
ax.barh(y + 0.2, mad["β regex v1 (titular)"], height=0.4, color=ROJO, label="queja explícita (regex v1)")
ax.barh(y - 0.2, mad[f"β {FINAL} (sensibilidad)"], height=0.4, color=GRIS, label=f"{FINAL} (sensibilidad)")
ax.set_yticks(y); ax.set_yticklabels([NOMBRE[e] for e in mad.index])
ax.set(xlabel="Estrellas por reseña con la falla (mismo restaurante)", title="Madrid: costo por etapa, titular y sensibilidad")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIGURAS / "07_madrid_titular_vs_sensibilidad.png", dpi=150); plt.show()

<Figure size 750x360 with 1 Axes>

#### 3.2 P2 · ¿Qué explica las reseñas negativas? Con control de circularidad

La razón "1-2★ contra 5★" de una etapa es en parte circular, porque una queja suena negativa por definición. Para saber si una etapa es **específica** de las malas experiencias, se compara con la razón de **cualquier oración negativa**, medida con un clasificador de tono entrenado con las estrellas (Madrid train).

In [8]:
o_mad = datos.cargar_oraciones("Madrid", con_resena=True)
rest_tr, _ = et.particion_restaurantes(o_mad.restaurante)
negativa = m.clasificador_negatividad(o_mad[o_mad.restaurante.isin(rest_tr)])

filas, rng = [], np.random.default_rng(0)
for c in CIUDADES:
    r0 = datos.cargar_resenas(c, ["review_id", "estrellas"])
    neg_r, cinco_r = r0.estrellas <= 2, r0.estrellas == 5
    o_c = datos.cargar_oraciones(c)
    muestra = np.concatenate([rng.choice(r0.review_id[neg_r], min(20_000, neg_r.sum()), replace=False),
                              rng.choice(r0.review_id[cinco_r], 20_000, replace=False)])
    o_s = o_c[o_c.review_id.isin(set(muestra))]
    tono = pd.Series(negativa(o_s.oracion), index=o_s.index).groupby(o_s.review_id.values).max()
    est = r0.set_index("review_id").estrellas.loc[tono.index]
    control = tono[est <= 2].mean() / tono[est == 5].mean()
    for nombre, fuente in FUENTES.items():
        r = por_resena(c, fuente)
        n_, c_ = r.estrellas <= 2, r.estrellas == 5
        fila = {"ciudad": c, "fuente": nombre, "tono negativo (control)": control,
                "% 1-2★ con falla del ciclo": r.loc[n_, et.CICLO].max(axis=1).mean() * 100,
                "% 1-2★ con falla fuera del ciclo": r.loc[n_, ["sabor", "precio", "local"]].max(axis=1).mean() * 100}
        for e in et.ETIQUETAS:
            fila[e] = r.loc[n_, e].mean() / max(r.loc[c_, e].mean(), 1e-9)
        filas.append(fila)
p2 = pd.DataFrame(filas).set_index(["fuente", "ciudad"])
display(p2.round(1))
p2_sobre_tono = p2[et.ETIQUETAS].div(p2["tono negativo (control)"], axis=0)
print("Veces que la razón de cada etapa supera la del tono negativo general:")
display(p2_sobre_tono.round(1))

,,tono negativo (control),% 1-2★ con falla del ciclo,% 1-2★ con falla fuera del ciclo,llegada,atencion,cocina_mesa,cobro,sabor,precio,local
fuente,ciudad,,,,,,,,,,
regex v1 (titular),Madrid,2.5,26.7,29.0,2.3,37.7,11.7,21.0,22.0,5.5,3.6
LogReg (sensibilidad),Madrid,2.5,82.9,83.8,2.2,5.6,5.0,6.8,4.3,3.3,1.7
regex v1 (titular),Barcelona,2.2,27.4,30.5,1.3,33.8,10.3,29.8,22.6,6.0,4.6
LogReg (sensibilidad),Barcelona,2.2,84.3,85.9,1.7,5.4,4.9,7.1,4.1,3.2,1.8
regex v1 (titular),London,2.0,33.0,29.2,2.4,26.4,12.5,11.9,20.4,4.1,3.7
LogReg (sensibilidad),London,2.0,87.6,84.5,2.9,4.6,5.3,7.3,4.0,3.2,2.2
regex v1 (titular),Paris,2.0,29.2,27.7,2.0,25.9,7.6,19.2,21.0,3.9,4.7
LogReg (sensibilidad),Paris,2.0,85.6,84.1,2.1,4.5,4.1,6.2,2.9,2.9,1.9
regex v1 (titular),New_York,1.9,25.1,31.4,1.9,24.1,11.1,19.0,18.0,4.0,2.5


Veces que la razón de cada etapa supera la del tono negativo general:


,,llegada,atencion,cocina_mesa,cobro,sabor,precio,local
fuente,ciudad,,,,,,,
regex v1 (titular),Madrid,0.9,15.4,4.8,8.6,9.0,2.2,1.5
LogReg (sensibilidad),Madrid,0.9,2.3,2.0,2.8,1.8,1.4,0.7
regex v1 (titular),Barcelona,0.6,15.3,4.7,13.5,10.2,2.7,2.1
LogReg (sensibilidad),Barcelona,0.8,2.4,2.2,3.2,1.9,1.4,0.8
regex v1 (titular),London,1.2,12.9,6.1,5.8,9.9,2.0,1.8
LogReg (sensibilidad),London,1.4,2.3,2.6,3.5,2.0,1.6,1.1
regex v1 (titular),Paris,1.0,13.2,3.9,9.8,10.7,2.0,2.4
LogReg (sensibilidad),Paris,1.0,2.3,2.1,3.2,1.5,1.5,1.0
regex v1 (titular),New_York,1.0,13.0,6.0,10.2,9.7,2.2,1.3


#### 3.3 P3 · ¿Las fallas separan a los restaurantes?

In [9]:
filas = []
for c in CIUDADES:
    for nombre, fuente in FUENTES.items():
        r = por_resena(c, fuente)
        n = r.groupby("restaurante").size(); pares = n[n >= 100].index
        rr = r[r.restaurante.isin(pares)]
        tasa = rr.groupby("restaurante")[et.ETIQUETAS].mean(); prom = rr.groupby("restaurante").estrellas.mean()
        filas.append({"ciudad": c, "fuente": nombre, "pares": len(pares), **{e: spearmanr(tasa[e], prom)[0] for e in et.ETIQUETAS}})
p3 = pd.DataFrame(filas).set_index(["fuente", "ciudad"])
display(p3.round(2))

# Control en Madrid: tasa de oraciones con tono negativo por restaurante
r = por_resena("Madrid", "regex_v1"); n = r.groupby("restaurante").size(); pares = n[n >= 100].index
om = o_mad[o_mad.restaurante.isin(pares)]
tono_rest = pd.Series(negativa(om.oracion), index=om.index).groupby(om.restaurante.values).mean()
prom = r[r.restaurante.isin(pares)].groupby("restaurante").estrellas.mean()
rho_tono = spearmanr(tono_rest.loc[prom.index], prom)[0]
print(f"Madrid, control: Spearman entre la tasa de oraciones con tono negativo y el promedio = {rho_tono:.2f}")

,,pares,llegada,atencion,cocina_mesa,cobro,sabor,precio,local
fuente,ciudad,,,,,,,,
regex v1 (titular),Madrid,366,-0.14,-0.71,-0.61,-0.37,-0.50,-0.61,-0.34
LogReg (sensibilidad),Madrid,366,-0.21,-0.76,-0.76,-0.59,-0.61,-0.71,-0.41
regex v1 (titular),Barcelona,914,-0.11,-0.75,-0.61,-0.49,-0.64,-0.67,-0.36
LogReg (sensibilidad),Barcelona,914,-0.16,-0.82,-0.80,-0.65,-0.73,-0.75,-0.31
regex v1 (titular),London,1679,-0.34,-0.59,-0.54,-0.40,-0.34,-0.37,-0.41
LogReg (sensibilidad),London,1679,-0.48,-0.62,-0.72,-0.45,-0.42,-0.47,-0.47
regex v1 (titular),Paris,1191,-0.14,-0.75,-0.54,-0.33,-0.37,-0.57,-0.36
LogReg (sensibilidad),Paris,1191,-0.11,-0.77,-0.72,-0.54,-0.50,-0.66,-0.37
regex v1 (titular),New_York,1086,-0.06,-0.39,-0.48,-0.22,-0.19,-0.24,-0.09


Madrid, control: Spearman entre la tasa de oraciones con tono negativo y el promedio = -0.92


#### 3.4 P4 · Saturación, por reseña y por oración

En el avance, "local lleno" se midió por reseña, y eso mezcla oraciones distintas. Aquí también se mide **dentro de la misma oración**.

In [10]:
rx_lleno = re.compile(lexico.LLENO)
filas = []
for c in CIUDADES:
    o_c = datos.cargar_oraciones(c)
    lleno_o = o_c.oracion.str.lower().str.contains(rx_lleno).values
    lleno_r = pd.Series(lleno_o).groupby(o_c.review_id.values).max()
    for nombre, fuente in FUENTES.items():
        pred = pd.read_parquet(sc.ruta_pred(fuente, c))
        fr = pred.groupby("review_id")[et.CICLO].max().loc[lleno_r.index]
        fila = {"ciudad": c, "fuente": nombre, "% reseñas que mencionan lleno": lleno_r.mean() * 100}
        for e in et.CICLO:
            fila[f"{e} reseña"] = fr.loc[lleno_r.values, e].mean() / fr.loc[~lleno_r.values, e].mean()
            fila[f"{e} oración"] = pred.loc[lleno_o, e].mean() / max(pred.loc[~lleno_o, e].mean(), 1e-9)
        filas.append(fila)
p4 = pd.DataFrame(filas).set_index(["fuente", "ciudad"])
p4.round(2)

,,% reseñas que mencionan lleno,llegada reseña,llegada oración,atencion reseña,atencion oración,cocina_mesa reseña,cocina_mesa oración,cobro reseña,cobro oración
fuente,ciudad,,,,,,,,,
regex v1 (titular),Madrid,10.74,3.56,6.45,1.19,0.87,1.64,2.16,1.08,0.14
LogReg (sensibilidad),Madrid,10.74,2.99,5.77,1.15,0.50,1.42,1.38,1.21,0.13
regex v1 (titular),Barcelona,9.59,3.44,6.14,1.37,1.11,1.79,3.00,1.21,0.30
LogReg (sensibilidad),Barcelona,9.59,3.12,5.84,1.24,0.59,1.55,1.70,1.23,0.14
regex v1 (titular),London,14.26,2.85,4.92,1.55,1.02,1.91,2.71,1.64,0.31
LogReg (sensibilidad),London,14.26,2.86,5.40,1.30,0.51,1.64,1.71,1.47,0.14
regex v1 (titular),Paris,10.43,2.75,4.28,1.21,0.96,1.89,2.83,1.13,0.30
LogReg (sensibilidad),Paris,10.43,2.60,4.88,1.20,0.54,1.51,1.76,1.25,0.12
regex v1 (titular),New_York,14.41,2.69,5.55,1.34,1.01,1.72,2.53,1.18,0.20


In [11]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, c in zip(axs, ["Madrid", "Barcelona"]):
    r = por_resena(c, "regex_v1")
    anual = (r.groupby(r.fecha.dt.year)[et.CICLO].mean() * 100).loc[2012:2019]
    for e, col in zip(et.CICLO, ["#9db7d5", ROJO, "#e3a33b", AZUL]):
        ax.plot(anual.index, anual[e], marker="o", ms=3, label=NOMBRE[e], color=col)
    ax.set(title=f"{c} · quejas explícitas", ylabel="% de reseñas con falla")
axs[0].legend(fontsize=8); plt.tight_layout(); plt.savefig(FIGURAS / "07_tendencia.png", dpi=150); plt.show()

<Figure size 1100x360 with 2 Axes>

### Lectura

**P1 · Costo de una queja explícita (titular).** Con el regex v1, la atención en sala es la falla más cara en las 6 ciudades: de −1.34★ (New Delhi) a −1.58★ (Madrid) por reseña, comparada con otras del mismo restaurante. Le siguen el cobro (de −0.80★ a −1.05★) y el pase (de −0.60★ a −0.86★); la llegada cuesta poco (de −0.08★ a −0.22★). Es una asociación dentro del restaurante, no un efecto causal.

**Análisis de sensibilidad con LogReg.** Con las predicciones de LogReg, los costos bajan a menos de la mitad (Atención de −0.40★ a −0.69★), y la atención sigue siendo la etapa más cara del ciclo en las 6 ciudades (el orden entre cobro y pase sí cambia según la ciudad). LogReg marca muchas reseñas que no tienen la falla (06), y su error no es parejo: sigue al tono negativo. Su β no se puede leer como un límite del costo real; solo muestra que el primer lugar de la atención no depende del detector.

**P2 · Con control de circularidad.** El tono negativo general (cualquier oración que suene negativa) es entre 1.9 y 2.5 veces más frecuente en reseñas de 1-2★ que en las de 5★. La queja explícita de atención separa **entre 11.7 y 15.4 veces más** que ese tono, en las 6 ciudades. El hallazgo del avance no es solo circularidad. Con LogReg, en cambio, la atención apenas supera al tono (de 1.9 a 2.4 veces), y eso es otra señal de que su error es diferencial. La llegada no separa más que el tono (de 0.6 a 1.2 veces). Una de cada cuatro reseñas de 1-2★ (del 21% al 33% según la ciudad) tiene una queja explícita del ciclo.

**P3 · Con control.** Entre los restaurantes con 100 reseñas o más, la tasa de quejas explícitas de atención se correlaciona con el promedio de estrellas (Spearman de −0.39 en New York a −0.75 en Barcelona y Paris; −0.71 en Madrid, frente a −0.72 en el avance). Pero el tono negativo general correlaciona todavía más (−0.92 en Madrid). **Esto debilita la afirmación del avance:** la atención es la etapa del ciclo que más separa restaurantes, pero no separa más que la negatividad en general. Lo que aporta es saber **qué** falla, no predecir mejor el promedio.

**P4 · Saturación.** Por reseña se reproduce el avance: con el local lleno, las quejas de llegada son 3.56 veces más frecuentes en Madrid y las del pase 1.64 veces; la atención casi no cambia (1.19). El patrón se repite en las 6 ciudades. Dentro de la misma oración el efecto es mayor (llegada 6.45×, pase 2.16×), pero ahí puede haber coocurrencia léxica: "busy" y "queue" o "wait" aparecen juntos en la misma frase. La conclusión del avance se sostiene en dirección; la magnitud depende de cómo se mida.

### Cifras clave


In [12]:
resultados.guardar("07", {"costos_regex_v1": {c: base.loc[c] for c in CIUDADES}, "r2": r2,
    "final": FINAL, "fuente_sensibilidad": FUENTE_M, "costos_sensibilidad": {c: con_modelo.loc[c] for c in CIUDADES},
    "p2": {f: p2.loc[f] for f in FUENTES}, "p2_sobre_tono": {f: p2_sobre_tono.loc[f] for f in FUENTES},
    "p3": {f: p3.loc[f] for f in FUENTES}, "p3_rho_tono_madrid": rho_tono, "p4": {f: p4.loc[f] for f in FUENTES}})

hallazgos_07.json: 10 claves → costos_regex_v1, costos_sensibilidad, final, fuente_sensibilidad, p2, p2_sobre_tono, p3, p3_rho_tono_madrid, p4, r2


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_07.json')

### Hallazgos

- **P1 (titular):** una queja explícita de atención cuesta de −1.34★ a −1.58★ en las 6 ciudades, la más cara del ciclo; Madrid reproduce el avance (−1.58). Con LogReg (sensibilidad) baja a −0.40★/−0.69★, y la atención sigue siendo la más cara en las 6 ciudades.
- **P2:** la queja explícita de atención separa malas de buenas reseñas entre 11.7 y 15.4 veces más que el tono negativo general: no es solo circularidad. Con LogReg, apenas unas 2 veces, porque su error sigue al tono.
- **P3 cambia de matiz:** la atención (−0.71 en Madrid) es la etapa del ciclo que más separa restaurantes, pero el tono negativo general separa más (−0.92).
- **P4 se sostiene:** con el local lleno, llegada 3.56× y pase 1.64× por reseña en Madrid; la atención casi no cambia. Por oración, el efecto es mayor pero puede incluir coocurrencia léxica.
- **El R² de 0.68 no se debe a fuga por autor:** con la partición por restaurante es 0.681; sacando del test a los autores que aparecen en train, 0.708.

### Verificación de citas

Toda oración citada entre comillas en este notebook tiene que existir literalmente en el corpus.

In [13]:
citas = []
verificadas = datos.verificar_citas(citas, ('Madrid',))
print(f"{len(verificadas)} citas verificadas:", verificadas)

0 citas verificadas: {}


## 08 · El producto: scorecard del ciclo de servicio

| | |
|---|---|
| **Qué entra** | predicciones por oración (`data/procesado/pred_{fuente}_{ciudad}.parquet`), costos por ciudad (07), precisión y recall del gold (06) |
| **Qué sale** | `figuras/08_scorecard_*.png` |

El gerente conoce su promedio de estrellas, pero no sabe **qué etapa** le falla, **cuánto** le cuesta ni **cómo está frente a sus pares**. El scorecard responde esas tres preguntas con evidencia textual.

**Qué mide.** El 06 concluyó que ningún modelo aprendido es apto, así que el scorecard usa el regex v1: **quejas explícitas (piso de la tasa real)**. Es preciso pero ve poco (recall de 0.08 en Atención), así que las tasas se leen como un piso, no como la tasa real. Se muestran crudas, con contracción y sin la corrección Rogan-Gladen: en las 4 etapas del ciclo el recall del regex es menor a 0.3 y la propia regla del scorecard la desactiva. En precio, fuera del ciclo, el recall queda apenas sobre el corte (0.31); tampoco se corrige, para no mezclar tasas corregidas y crudas en la misma tabla. LogReg aparece solo como columna secundaria, "señal amplia, no validada", sin percentil. Llegada va marcada como no validada.

**Supuesto no verificado:** que el sesgo de recall del regex sea parejo entre restaurantes. Si un restaurante recibe quejas más parafraseadas que otro, su percentil queda sesgado.

**Estado:** final. Producto = regex v1 (quejas explícitas, piso de la tasa real), porque ningún modelo aprendido es apto (06). LogReg va como columna secundaria no validada.

In [1]:
from foodie_advisor import FIGURAS, PROCESADO, datos, etiquetas as et, scorecard as sc, resultados

h06, h07 = resultados.cargar("06"), resultados.cargar("07")
# El producto lo fija el 06: ningún modelo aprendido es apto → regex v1 (quejas explícitas)
assert h06["producto"] == "Regex v1", h06["producto"]
FUENTE = "regex_v1"
ROTULO = "quejas explícitas (piso de la tasa real)"
# Sin Rogan-Gladen. En las 4 etapas del ciclo el recall del regex en el gold es < 0.3 y la regla del scorecard
# lo desactiva. En precio (fuera del ciclo) es apenas mayor; tampoco se corrige, para no mezclar tasas
# corregidas y crudas en la misma tabla. Se declara.
rec_rx = {e: v["recall (ponderado)"] for e, v in h06["precision_regex"].items()}
assert all(rec_rx[e] < sc.MIN_RECALL_RG for e in et.CICLO), rec_rx
sobre_corte = {e: round(v, 2) for e, v in rec_rx.items() if v >= sc.MIN_RECALL_RG}
MEDICION = None
SECUNDARIA = ("LogReg", h07["fuente_sensibilidad"])
NOTAS = {"llegada": f"no validada: {h06['positivos_gold']['llegada']} positivos en el gold"}
costos = pd.read_csv(PROCESADO / "costos_regex_v1.csv", index_col=[0, 1])
costos_todas = costos
from foodie_advisor import CIUDADES as CIUDADES_TODAS
print(f"Fuente: {FUENTE} ({ROTULO}) | sin Rogan-Gladen (recall del regex en el gold: {', '.join(f'{e} {v:.2f}' for e, v in rec_rx.items())})")
print(f"Columna secundaria: {SECUNDARIA[0]} — señal amplia, no validada, sin percentil")
if sobre_corte:
    print(f"Excepción declarada: recall ≥ {sc.MIN_RECALL_RG} en {sobre_corte}; igual se muestra crudo, sin corregir.")

ctx = {}
for c in ["Madrid", "Barcelona"]:
    sc.predecir_regex(c)
    ctx[c] = sc.contexto_ciudad(c, FUENTE, costos.loc[c, "β"], MEDICION, secundaria=SECUNDARIA, notas=NOTAS, rotulo=ROTULO)
    print(f"{c}: {len(ctx[c]['pares'])} restaurantes pares (100+ reseñas)")

Fuente: regex_v1 (quejas explícitas (piso de la tasa real)) | sin Rogan-Gladen (recall del regex en el gold: atencion 0.08, cobro 0.02, cocina_mesa 0.10, llegada 0.00, local 0.06, precio 0.31, sabor 0.19)
Columna secundaria: LogReg — señal amplia, no validada, sin percentil
Excepción declarada: recall ≥ 0.3 en {'precio': 0.31}; igual se muestra crudo, sin corregir.


Madrid: 366 restaurantes pares (100+ reseñas)


Barcelona: 914 restaurantes pares (100+ reseñas)


### Cómo se calcula cada número

| Columna | Qué es |
|---|---|
| **quejas explícitas (piso de la tasa real)** | parte de sus reseñas con al menos una queja explícita en esa etapa (regex del avance). Se contrae hacia la media de los pares con un prior beta ajustado a los pares de su ciudad, para que un restaurante con pocas reseñas no salga extremo por azar. IC 95% de la posterior. |
| **percentil** | dónde queda frente a sus pares (100 = el que más falla). |
| **costo por falla** | estrellas que cuesta una reseña con esa falla frente a otra del mismo restaurante (efectos fijos, 07). |
| **estrellas en juego** | tasa × costo: cuántas estrellas de su promedio se asocian con esa etapa. **Es asociación, no la ganancia esperada por arreglar.** |
| **acción prioritaria** | exceso = (tasa contraída − mediana de la ciudad) × \|β\|. Son candidatas las etapas del ciclo con ≥ 5 reseñas marcadas, percentil ≥ 75 y límite inferior del intervalo creíble al 80% por encima de la mediana. Se elige la de mayor exceso. Si no hay candidatas y sabor, precio o local está en percentil ≥ 75: "el problema está fuera del ciclo". Si tampoco: "sin punto débil diferencial". No se usa tasa × β, porque la atención ganaría siempre. |
| **sello** | Atención: validada (6/6 en el gold). Cocina → mesa: señal débil (2/8). Cobro y Llegada: no validadas. |

In [2]:
def mostrar(nombre, ciudad, archivo):
    s = sc.scorecard(nombre, ctx[ciudad])
    print(f"{nombre} ({ciudad}): {s['reseñas']:,} reseñas | {s['promedio']:.2f}★ (mediana de pares {s['mediana pares ★']:.2f}★) | "
          f"en juego: ciclo {s['en juego ciclo']:.3f}★, fuera del ciclo {s['en juego fuera']:.3f}★")
    print("ACCIÓN PRIORITARIA:", s["accion"])
    display(s["tabla"].drop(columns="nombre").round(2))
    for a in s["avisos"]:
        print("  aviso:", a)
    sc.figura(s, FIGURAS / archivo); plt.show()
    return s

botin = mostrar("Restaurante_Botin", "Madrid", "08_scorecard_botin.png")

Restaurante_Botin (Madrid): 3,642 reseñas | 3.96★ (mediana de pares 4.28★) | en juego: ciclo -0.060★, fuera del ciclo -0.154★
ACCIÓN PRIORITARIA: Atacar primero: 4. Cobro — no validada


,reseñas con falla,% observado,quejas explícitas (piso de la tasa real),corregida (aprox.),IC inf %,IC sup %,mediana pares %,percentil,costo por falla ★,estrellas en juego,nota,"% LogReg (señal amplia, no validada)",exceso ★ (vs mediana),inf80 %,candidata,sello,% estimado
etapa,,,,,,,,,,,,,,,,,
llegada,40,1.10,1.10,False,0.78,1.46,0.53,77.32,-0.20,-0.00,no validada: 6 positivos en el gold,13.98,0.00,0.88,True,no validada,1.10
atencion,84,2.31,2.31,False,1.85,2.82,2.28,51.91,-1.58,-0.04,,30.51,0.00,2.00,False,validada (6/6 en el gold),2.31
cocina_mesa,50,1.37,1.39,False,1.03,1.79,2.07,29.51,-0.75,-0.01,,14.74,-0.01,1.15,False,señal débil (2/8 en el gold),1.39
cobro,40,1.10,1.08,False,0.77,1.43,0.49,89.34,-0.98,-0.01,,7.39,0.01,0.87,True,no validada,1.08
sabor,184,5.05,4.99,False,4.30,5.70,2.09,94.26,-1.18,-0.06,,34.05,0.03,4.54,False,,4.99
precio,513,14.09,14.02,False,12.91,15.15,5.25,95.36,-0.62,-0.09,,46.02,0.05,13.29,False,,14.02
local,82,2.25,2.24,False,1.78,2.73,1.38,77.05,-0.39,-0.01,,17.55,0.00,1.93,False,,2.24


<Figure size 1100x640 with 3 Axes>

In [3]:
ctapas = mostrar("C_Tapas", "Madrid", "08_scorecard_ctapas.png")

C_Tapas (Madrid): 337 reseñas | 2.66★ (mediana de pares 4.28★) | en juego: ciclo -0.240★, fuera del ciclo -0.139★
ACCIÓN PRIORITARIA: Atacar primero: 2. Atención en sala — validada (6/6 en el gold)


,reseñas con falla,% observado,quejas explícitas (piso de la tasa real),corregida (aprox.),IC inf %,IC sup %,mediana pares %,percentil,costo por falla ★,estrellas en juego,nota,"% LogReg (señal amplia, no validada)",exceso ★ (vs mediana),inf80 %,candidata,sello,% estimado
etapa,,,,,,,,,,,,,,,,,
llegada,3,0.89,0.89,False,0.20,2.06,0.53,69.13,-0.20,-0.00,no validada: 6 positivos en el gold,10.98,0.00,0.36,False,no validada,0.89
atencion,36,10.68,9.63,False,6.88,12.74,2.28,98.91,-1.58,-0.15,,47.18,0.12,7.76,True,validada (6/6 en el gold),9.63
cocina_mesa,38,11.28,10.00,False,7.21,13.15,2.07,99.45,-0.75,-0.07,,33.83,0.06,8.12,True,señal débil (2/8 en el gold),10.00
cobro,5,1.48,1.19,False,0.44,2.28,0.49,92.08,-0.98,-0.01,,15.73,0.01,0.63,True,no validada,1.19
sabor,11,3.26,3.08,False,1.65,4.88,2.09,77.87,-1.18,-0.04,,35.31,0.01,2.07,False,,3.08
precio,54,16.02,15.16,False,11.66,19.00,5.25,96.72,-0.62,-0.09,,49.26,0.06,12.82,False,,15.16
local,8,2.37,2.23,False,1.04,3.83,1.38,76.50,-0.39,-0.01,,12.17,0.00,1.37,False,,2.23


<Figure size 1100x640 with 3 Axes>

### Dos promedios bajos, dos problemas distintos

In [4]:
comp = pd.DataFrame({s["restaurante"]: s["tabla"]["estrellas en juego"] for s in [botin, ctapas]})
comp.index = [et.NOMBRE[e] for e in comp.index]
comp.loc["Total ciclo"] = [s["en juego ciclo"] for s in [botin, ctapas]]
comp.loc["Total fuera del ciclo"] = [s["en juego fuera"] for s in [botin, ctapas]]
comp.round(3)

,Restaurante_Botin,C_Tapas
1. Llegada y mesa,-0.002,-0.002
2. Atención en sala,-0.036,-0.152
3. Cocina → mesa,-0.010,-0.075
4. Cobro,-0.011,-0.012
Fuera del ciclo: cocina,-0.059,-0.036
Fuera del ciclo: precio,-0.086,-0.093
Fuera del ciclo: local,-0.009,-0.009
Total ciclo,-0.060,-0.240
Total fuera del ciclo,-0.154,-0.139


Los dos tienen promedios bajos frente a la mediana de sus 366 pares (4.28★), pero por razones distintas, y eso es justo lo que un promedio no muestra.

- **C_Tapas (2.66★):** la queja explícita de atención aparece en el 9.6% de sus reseñas (percentil 99) y la del pase en el 10.0% (percentil 99). El ciclo de servicio se asocia con 0.24★ de su promedio. **Acción prioritaria: atención en sala (validada).**
- **Botín (3.96★):** la atención (percentil 52) y el pase (percentil 30) están en rango normal. Lo que más destaca frente a sus pares está fuera del ciclo: precio (percentil 95) y sabor (percentil 94). Dentro del ciclo, el cobro está en el percentil 89 con 40 reseñas marcadas, y por la regla esa es su acción prioritaria. Pero **el cobro no está validado** en el gold, y una de sus citas (*"we wernt charged for it either"*) muestra que el regex también marca menciones que no son quejas. Para Botín, la lectura honesta es que su diferencia frente a los pares está en el precio y el sabor, y que la señal de cobro hay que revisarla a mano.

En el avance, Botín tenía la atención en el percentil 54 y el pase en el 37; ahora quedan en el 52 y el 30, por la contracción bayesiana y porque una reseña cuenta como "con falla" si alguna de sus oraciones la tiene.

### Un caso de Barcelona

Se elige con una regla fija, no a mano: entre los restaurantes de Barcelona con 300 reseñas o más, el que tiene más estrellas en juego en el ciclo de servicio.

In [5]:
b = ctx["Barcelona"]
grandes = b["n"][b["n"] >= 300].index
en_juego = (b["tasa_contraida"].loc[grandes, et.CICLO] * b["costo"][et.CICLO]).sum(axis=1)
caso_bcn = en_juego.idxmin()
bcn = mostrar(caso_bcn, "Barcelona", "08_scorecard_barcelona.png")

El_Cafe_D_en_Victor (Barcelona): 361 reseñas | 2.78★ (mediana de pares 4.31★) | en juego: ciclo -0.306★, fuera del ciclo -0.160★
ACCIÓN PRIORITARIA: Atacar primero: 2. Atención en sala — validada (6/6 en el gold)


,reseñas con falla,% observado,quejas explícitas (piso de la tasa real),corregida (aprox.),IC inf %,IC sup %,mediana pares %,percentil,costo por falla ★,estrellas en juego,nota,"% LogReg (señal amplia, no validada)",exceso ★ (vs mediana),inf80 %,candidata,sello,% estimado
etapa,,,,,,,,,,,,,,,,,
llegada,2,0.55,0.59,False,0.08,1.57,0.59,50.11,-0.08,-0.00,no validada: 6 positivos en el gold,8.59,0.00,0.18,False,no validada,0.59
atencion,65,18.01,16.29,False,12.84,20.04,2.33,99.89,-1.57,-0.26,,55.96,0.22,13.98,True,validada (6/6 en el gold),16.29
cocina_mesa,17,4.71,4.39,False,2.66,6.46,2.39,83.37,-0.74,-0.03,,27.42,0.01,3.18,True,señal débil (2/8 en el gold),4.39
cobro,7,1.94,1.68,False,0.70,3.05,0.31,94.09,-1.05,-0.02,,20.78,0.01,0.96,True,no validada,1.68
sabor,20,5.54,5.04,False,3.18,7.23,2.23,92.23,-1.21,-0.06,,42.66,0.03,3.75,False,,5.04
precio,55,15.24,14.70,False,11.32,18.42,5.01,94.09,-0.61,-0.09,,51.25,0.06,12.43,False,,14.70
local,6,1.66,1.63,False,0.70,2.93,1.31,67.07,-0.51,-0.01,,11.36,0.00,0.94,False,,1.63


<Figure size 1100x640 with 3 Axes>

### Distribución de la acción prioritaria por ciudad

Si más del 80% de los restaurantes cae en la misma etapa, la regla no discrimina.

In [6]:
dist = {}
for c in CIUDADES_TODAS:
    sc.predecir_regex(c)
    cx = sc.contexto_ciudad(c, FUENTE, costos_todas.loc[c, "β"]) if c not in ctx else ctx[c]
    dist[c] = cx["prioridad"].value_counts(normalize=True) * 100
dist = pd.DataFrame(dist).fillna(0).round(1)
display(dist)
max_ciclo = dist.loc[[e for e in et.CICLO if e in dist.index]].max()
print("¿La regla discrimina? (ninguna etapa ≥ 80% en ninguna ciudad):", bool((dist.max() < 80).all()))

,Madrid,Barcelona,London,Paris,New_York,New_Delhi
atencion,22.7,23.6,14.0,24.1,9.8,9.0
cobro,3.0,2.8,12.9,0.5,1.9,4.2
cocina_mesa,10.1,8.4,15.1,7.8,15.3,14.8
fuera: local,6.8,5.6,5.1,7.6,13.4,8.7
fuera: precio,9.6,6.6,7.3,7.7,9.3,7.7
fuera: sabor,8.2,6.0,7.1,10.0,7.6,9.4
llegada,7.9,9.2,8.1,7.1,11.7,7.4
sin punto débil diferencial,31.7,37.7,30.3,35.3,31.0,38.7


¿La regla discrimina? (ninguna etapa ≥ 80% en ninguna ciudad): True


In [7]:
# Las citas del scorecard salen del corpus por construcción; igual se verifican
todas = botin["citas"] + ctapas["citas"] + bcn["citas"]
print(len(datos.verificar_citas(todas, ("Madrid", "Barcelona"))), "citas del scorecard verificadas")

9 citas del scorecard verificadas


### Cifras clave


In [8]:
from foodie_advisor import resultados
def resumen_sc(s):
    return {"restaurante": s["restaurante"], "ciudad": s["ciudad"], "resenas": s["reseñas"], "promedio": s["promedio"],
            "mediana_pares": s["mediana pares ★"], "n_pares": s["n pares"], "en_juego_ciclo": s["en juego ciclo"],
            "en_juego_fuera": s["en juego fuera"], "peor_etapa": s["peor etapa"], "citas": s["citas"],
            "prioridad": s["prioridad"], "accion": s["accion"],
            "tabla": s["tabla"][["% observado", "% estimado", "corregida (aprox.)", "percentil", "estrellas en juego", "nota"]
                                + [c for c in s["tabla"].columns if "no validada" in c]], "rotulo": s["rotulo"]}
resultados.guardar("08", {"fuente": FUENTE, "corregido": MEDICION is not None, "avisos": botin["avisos"],
                          "distribucion_prioridad": dist, "discrimina": bool((dist.max() < 80).all()), "sobre_corte_rg": sobre_corte,
                          "botin": resumen_sc(botin), "ctapas": resumen_sc(ctapas), "barcelona": resumen_sc(bcn)})

hallazgos_08.json: 9 claves → avisos, barcelona, botin, corregido, ctapas, discrimina, distribucion_prioridad, fuente, sobre_corte_rg


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_08.json')

### Hallazgos

- **El producto usa quejas explícitas (regex v1), un piso de la tasa real,** porque ningún modelo aprendido fue apto en el gold (06). LogReg aparece solo como señal secundaria no validada y sin percentil.
- **La acción prioritaria discrimina:** se elige por exceso frente a la mediana de los pares, no por estrellas en juego. En las 6 ciudades, atención es la prioridad de entre el 9.0% y el 24.1% de los restaurantes, y entre el 30.3% y el 38.7% no tiene un punto débil diferencial.
- **C_Tapas (2.66★) y El Cafe D'en Victor (Barcelona, 2.78★):** atención en sala, validada, en el percentil 99 y 99.9.
- **Botín (3.96★):** su diferencia está fuera del ciclo (precio p95, sabor p94). La regla le asigna cobro (p89), una etapa **no validada**, y la evidencia textual muestra ruido.
- **Estrellas en juego = asociación, no la ganancia esperada por arreglar.** El 09 muestra que el scorecard describe pero no predice el promedio futuro.

### Verificación de citas

Toda oración citada entre comillas en este notebook tiene que existir literalmente en el corpus.

In [9]:
citas = ['we wernt charged for it either']
verificadas = datos.verificar_citas(citas, ('Madrid', 'Barcelona'))
print(f"{len(verificadas)} citas verificadas:", verificadas)

1 citas verificadas: {'we wernt charged for it either': 'Madrid'}


## 09 · Validación temporal: ¿el scorecard predice y apunta?

| | |
|---|---|
| **Qué entra** | reseñas de las 6 ciudades, predicciones por oración del regex v1 y de LogReg (07) |
| **Qué sale** | `resultados/hallazgos_09.json`: ΔR² por fuente y ciudad, coincidencia de la prioridad, ganadores |

## Reglas (escritas antes de correr; el notebook se corre una sola vez)

- **Muestra:** restaurantes con ≥ 50 reseñas hasta 2017 y ≥ 30 en 2018-2019. Se excluye 2020. **Sesgo de supervivencia:** solo entran los restaurantes que siguieron abiertos y reseñados; los que cerraron, quizá por sus fallas, no están.
- **Resultado:** promedio de estrellas en 2018-2019.
- **Base:** promedio contraído hasta 2017 + log(n de reseñas hasta 2017).
- **Fuentes**, cada una añadida a la base por separado:
  - (a) estrellas en juego con el regex v1;
  - (b) estrellas en juego con LogReg;
  - (c) predicción de Ridge sobre TF-IDF.
- **Especificación de (a) y (b),** fijada en Madrid: dos variables, las estrellas en juego del ciclo y las de fuera del ciclo, sumadas por grupo. Con las 7 etapas por separado se sobreajustaba.
- **Sin fuga temporal:** los β de (a) y (b) y el modelo Ridge se estiman **solo con reseñas hasta 2017**. Ridge, además, con validación cruzada por restaurante: el valor de cada restaurante sale de un modelo que no vio sus reseñas.
- **Predecir:** ΔR² fuera de muestra sobre la base (5 particiones por restaurante), con IC bootstrap al 95%.
- **Apuntar:** coincidencia de la etapa prioritaria (regla del 08) entre hasta-2017 y 2018-2019, entre los restaurantes con prioridad del ciclo hasta 2017. Se compara con una permutación (p95) y con el baseline "siempre atención". No se prueba "empeoran más en X", porque la regresión a la media invalida esa comparación.
- **Ciudades:** se ajusta en Madrid, se decide en Barcelona; London, Paris, New York y New Delhi solo se reportan.
- **Ganador para PREDECIR:** una fuente gana si su IC de ΔR² excluye 0 en Barcelona (con ΔR² > 0) y el signo se repite en al menos 3 de las otras 4 ciudades. Si ganan varias, la de mayor ΔR² en Barcelona. Si no gana ninguna: "el scorecard describe, no predice".
- **Ganador para APUNTAR** (solo regex o LogReg): en Barcelona, la coincidencia supera el p95 de la permutación y el baseline "siempre atención", y eso se repite en al menos 3 de las otras 4 ciudades. Si ganan las dos, la de mayor margen sobre "siempre atención" en Barcelona. Si no gana ninguna: "la prioridad no es estable en el tiempo".

In [1]:
from foodie_advisor import CIUDADES, FIGURAS, etiquetas as et, resultados, temporal as tp

FUENTES = {"regex v1": "regex_v1", "LogReg": "LogReg_borrado"}
OTRAS = ["London", "Paris", "New_York", "New_Delhi"]

def dos_grupos(F):
    return pd.DataFrame({"en_juego_ciclo": F[[f"en_juego_{e}" for e in et.CICLO]].sum(axis=1),
                         "en_juego_fuera": F[[f"en_juego_{e}" for e in ["sabor", "precio", "local"]]].sum(axis=1)})

### 1. Predecir: ΔR² sobre la base, por fuente y ciudad

In [2]:
filas_r2, filas_ap, muestras = [], [], {}
for c in CIUDADES:
    tb = tp.tablas(c)
    muestras[c] = tb["n"]
    extras = {"regex v1": dos_grupos(tb["feats"]["regex_v1"]), "LogReg": dos_grupos(tb["feats"]["LogReg_borrado"]),
              "Ridge TF-IDF": tp.feature_ridge(tb)}
    for nombre, X in extras.items():
        filas_r2.append({"ciudad": c, "fuente": nombre, "restaurantes": tb["n"], **tp.delta_r2(tb["base"], X, tb["y"])})
    for nombre, f in FUENTES.items():
        filas_ap.append({"ciudad": c, "fuente": nombre, **tp.coincidencia(tb["prior"][f]["prio_antes"], tb["prior"][f]["prio_despues"])})
r2 = pd.DataFrame(filas_r2).set_index(["fuente", "ciudad"])
ap = pd.DataFrame(filas_ap).set_index(["fuente", "ciudad"])
print("Restaurantes en la muestra:", muestras)
r2.round(4)

Restaurantes en la muestra: {'Madrid': 253, 'Barcelona': 666, 'London': 1485, 'Paris': 810, 'New_York': 879, 'New_Delhi': 209}


,,restaurantes,R2 base,R2 con fuente,ΔR2,IC inf,IC sup
fuente,ciudad,,,,,,
regex v1,Madrid,253,0.6493,0.6488,-0.0006,-0.0125,0.0117
LogReg,Madrid,253,0.6493,0.6487,-0.0006,-0.0109,0.0111
Ridge TF-IDF,Madrid,253,0.6493,0.6491,-0.0002,-0.0029,0.0024
regex v1,Barcelona,666,0.6840,0.6824,-0.0016,-0.0096,0.0064
LogReg,Barcelona,666,0.6840,0.6785,-0.0055,-0.0094,-0.0013
Ridge TF-IDF,Barcelona,666,0.6840,0.6816,-0.0024,-0.0055,0.0007
regex v1,London,1485,0.2841,0.2807,-0.0035,-0.0091,0.0020
LogReg,London,1485,0.2841,0.2928,0.0087,-0.0015,0.0193
Ridge TF-IDF,London,1485,0.2841,0.2896,0.0054,-0.0015,0.0121


### 2. Apuntar: ¿la etapa prioritaria se repite?

In [3]:
ap.round(3)

,,restaurantes con prioridad del ciclo (antes),coincidencia,permutación media,permutación p95,siempre atención
fuente,ciudad,,,,,
regex v1,Madrid,107,0.271,0.064,0.103,0.206
LogReg,Madrid,139,0.453,0.151,0.194,0.345
regex v1,Barcelona,293,0.307,0.087,0.113,0.256
LogReg,Barcelona,341,0.496,0.160,0.191,0.378
regex v1,London,704,0.287,0.083,0.098,0.142
LogReg,London,744,0.392,0.132,0.151,0.331
regex v1,Paris,306,0.392,0.109,0.134,0.350
LogReg,Paris,391,0.463,0.154,0.182,0.391
regex v1,New_York,333,0.237,0.054,0.075,0.081


In [4]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for i, (fuente, col) in enumerate(zip(["regex v1", "LogReg", "Ridge TF-IDF"], ["#b5483a", "#8c8c8c", "#3b6ea5"])):
    d = r2.loc[fuente].loc[CIUDADES]
    x = np.arange(len(CIUDADES)) + (i - 1) * 0.25
    axs[0].errorbar(x, d["ΔR2"], yerr=[d["ΔR2"] - d["IC inf"], d["IC sup"] - d["ΔR2"]], fmt="o", color=col, label=fuente, capsize=2)
axs[0].axhline(0, color="black", lw=0.8); axs[0].set_xticks(range(len(CIUDADES))); axs[0].set_xticklabels(CIUDADES, rotation=20)
axs[0].set(title="Predecir 2018-2019: ΔR² sobre la base", ylabel="ΔR² (IC 95%)"); axs[0].legend(fontsize=8)
for i, (fuente, col) in enumerate(zip(["regex v1", "LogReg"], ["#b5483a", "#8c8c8c"])):
    d = ap.loc[fuente].loc[CIUDADES]
    x = np.arange(len(CIUDADES)) + (i - 0.5) * 0.3
    axs[1].bar(x, d["coincidencia"] * 100, width=0.28, color=col, label=f"{fuente}: coincidencia")
    axs[1].scatter(x, d["permutación p95"] * 100, marker="_", s=200, color="black")
    axs[1].scatter(x, d["siempre atención"] * 100, marker="x", color=col)
axs[1].set_xticks(range(len(CIUDADES))); axs[1].set_xticklabels(CIUDADES, rotation=20)
axs[1].set(title="Apuntar: ¿la prioridad hasta 2017 se repite en 2018-2019?", ylabel="% de restaurantes")
axs[1].legend(fontsize=8, title="— p95 permutación   × siempre atención", title_fontsize=7)
plt.tight_layout(); plt.savefig(FIGURAS / "09_validacion_temporal.png", dpi=150); plt.show()

<Figure size 1200x400 with 2 Axes>

### 3. Aplicar las reglas

In [5]:
def gana_predecir(f):
    b = r2.loc[(f, "Barcelona")]
    if not (b["IC inf"] > 0 and b["ΔR2"] > 0):
        return False
    return sum(r2.loc[(f, c), "ΔR2"] > 0 for c in OTRAS) >= 3

def gana_apuntar(f):
    ok = lambda c: ap.loc[(f, c), "coincidencia"] > max(ap.loc[(f, c), "permutación p95"], ap.loc[(f, c), "siempre atención"])
    return ok("Barcelona") and sum(ok(c) for c in OTRAS) >= 3

pred_ganan = [f for f in ["regex v1", "LogReg", "Ridge TF-IDF"] if gana_predecir(f)]
ap_ganan = [f for f in FUENTES if gana_apuntar(f)]
GANA_PREDECIR = max(pred_ganan, key=lambda f: r2.loc[(f, "Barcelona"), "ΔR2"]) if pred_ganan else None
GANA_APUNTAR = (max(ap_ganan, key=lambda f: ap.loc[(f, "Barcelona"), "coincidencia"] - ap.loc[(f, "Barcelona"), "siempre atención"])
                if ap_ganan else None)
FRASE_PREDECIR = f"Para predecir gana {GANA_PREDECIR}." if GANA_PREDECIR else "El scorecard describe, no predice."
FRASE_APUNTAR = f"Para apuntar gana {GANA_APUNTAR}." if GANA_APUNTAR else "La prioridad no es estable en el tiempo."
print("PREDECIR:", FRASE_PREDECIR, "| candidatas que cumplen:", pred_ganan)
print("APUNTAR:", FRASE_APUNTAR, "| candidatas que cumplen:", ap_ganan)

PREDECIR: El scorecard describe, no predice. | candidatas que cumplen: []
APUNTAR: Para apuntar gana LogReg. | candidatas que cumplen: ['regex v1', 'LogReg']


### Lectura

**Predecir: el scorecard describe, no predice.** En Barcelona, la base (promedio contraído hasta 2017 + log n) ya explica el 68.4% de la varianza del promedio 2018-2019. Ninguna fuente le agrega nada que se distinga de cero: el regex da ΔR² de −0.0016 (IC de −0.0096 a 0.0064), LogReg −0.0055 (IC de −0.0094 a −0.0013; resta) y Ridge −0.0024 (IC de −0.0055 a 0.0007). En las otras 4 ciudades el signo es mixto y todos los IC incluyen el 0, salvo LogReg en New Delhi, que también resta. Saber qué etapa falla no mejora el pronóstico del promedio futuro, más allá de lo que ya dice el promedio pasado.

**Apuntar: la prioridad sí es estable.** Entre los restaurantes de Barcelona que tenían una prioridad del ciclo hasta 2017, la misma etapa vuelve a ser la prioritaria en 2018-2019 en el 30.7% de los casos con el regex, frente al 11.3% del p95 de una permutación y al 25.6% de "siempre atención". Con LogReg, en el 49.6%, frente a 19.1% y 37.8%. Las dos fuentes cumplen la regla en Barcelona y en las 4 ciudades de reporte, y por margen sobre "siempre atención" gana **LogReg**.

**Salvedad importante.** LogReg no pasó la aptitud en el gold (06): su precisión es baja y su error sigue al tono negativo general (07). Que su prioridad sea estable puede reflejar un tono persistente del restaurante y no una falla concreta persistente. El regex, que sí es preciso, también pasa la prueba de apuntar, con un margen menor. Por eso el producto se queda con el regex.

**Sesgo de supervivencia:** solo entran los restaurantes con ≥ 50 reseñas hasta 2017 y ≥ 30 en 2018-2019 (666 en Barcelona). Los que cerraron no están, y entre ellos podrían estar los que más fallaban.

In [6]:
resultados.guardar("09", {"muestras": muestras, "r2": {f: r2.loc[f] for f in r2.index.get_level_values(0).unique()},
                          "apuntar": {f: ap.loc[f] for f in FUENTES}, "gana_predecir": GANA_PREDECIR, "gana_apuntar": GANA_APUNTAR,
                          "frase_predecir": FRASE_PREDECIR, "frase_apuntar": FRASE_APUNTAR,
                          "especificacion": "estrellas en juego del ciclo y de fuera del ciclo (2 variables), fijada en Madrid"})

hallazgos_09.json: 8 claves → apuntar, especificacion, frase_apuntar, frase_predecir, gana_apuntar, gana_predecir, muestras, r2


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_09.json')

### Hallazgos

- **El scorecard describe, no predice.** Ninguna fuente (regex, LogReg ni Ridge) mejora el pronóstico del promedio 2018-2019 sobre el promedio contraído hasta 2017: en Barcelona, ΔR² de −0.0016 (regex), −0.0055 (LogReg) y −0.0024 (Ridge). Ningún IC queda por encima de 0.
- **La prioridad sí es estable en el tiempo:** en Barcelona, la etapa prioritaria hasta 2017 se repite en 2018-2019 en el 30.7% de los restaurantes con el regex (p95 de la permutación: 11.3%; "siempre atención": 25.6%) y en el 49.6% con LogReg (19.1% y 37.8%). Se repite en las 4 ciudades de reporte.
- **Por la regla, para apuntar gana LogReg, pero no es apto (06)** y su estabilidad puede ser tono persistente. El regex también cumple, y es el que usa el producto.
- **Sesgo de supervivencia:** solo restaurantes que siguieron abiertos (de 209 en New Delhi a 1,485 en London).

## El producto: la app del gerente

El scorecard de la sección 08 se usa a través de una app de Streamlit (`producto/app.py`; la lógica está en `foodie_advisor/producto.py`). No entrena ni anota nada: reutiliza las predicciones del regex v1 y de LogReg y la misma regla de prioridad.

| Vista | Qué hace |
|---|---|
| **Mi restaurante** | ficha por etapa del ciclo frente a los pares de la ciudad: % de quejas con IC, percentil, estrellas en juego, acción prioritaria, tendencia por año y citas reales |
| **Reseñas nuevas** | se pegan reseñas o se sube un CSV de un local que no está en el dataset; se aplican el regex y LogReg y la misma regla de prioridad (con ≥ 30 reseñas) |
| **Mi ciudad** | ranking de restaurantes por etapa |

Cada queja se clasifica según las fuentes que la marcan: **confirmada** (regex y LogReg), **explícita** (solo regex; es la que cuenta en la tasa, como en el scorecard) y **posible** (solo LogReg; se muestra, pero no se cuenta).

Para correrla, desde la raíz del repo:

```bash
.venv/bin/streamlit run producto/app.py
```

La celda siguiente arma la misma ficha que muestra la vista **Mi restaurante** para C_Tapas (Madrid).

In [14]:
from foodie_advisor import producto

ctx_madrid = producto.cargar_ciudad("Madrid")
ficha_ct = producto.ficha(ctx_madrid, "C_Tapas")
print(f"{ficha_ct['nombre']}: {ficha_ct['n']} reseñas, {ficha_ct['estrellas']:.2f}★ "
      f"(mediana de {ficha_ct['n pares']} pares: {ficha_ct['mediana pares ★']:.2f}★) | prioridad: {ficha_ct['prioridad']}")
ficha_ct["tabla"].drop(columns=["ciclo"]).round(2)

C Tapas: 337 reseñas, 2.66★ (mediana de 366 pares: 4.28★) | prioridad: atencion


,Etapa,reseñas con queja,confirmadas,posibles (solo LogReg),% quejas,IC inf,IC sup,mediana pares %,percentil,exceso ★,candidata,costo ★,sello
etapa,,,,,,,,,,,,,
llegada,Llegada y mesa,3,3,34,0.89,0.21,2.05,0.53,69.13,0.00,False,-0.20,no validada
atencion,Atención en sala,36,36,123,9.63,6.90,12.75,2.28,98.91,0.12,True,-1.58,validada
cocina_mesa,Cocina → mesa,38,38,76,10.00,7.25,13.15,2.07,99.45,0.06,True,-0.75,señal débil
cobro,Cobro,5,5,48,1.19,0.44,2.29,0.49,91.80,0.01,True,-0.98,no validada
sabor,Sabor,11,11,108,3.08,1.67,4.91,2.09,77.87,0.01,False,-1.18,fuera del ciclo
precio,Precio,54,54,112,15.16,11.69,18.98,5.25,96.72,0.06,False,-0.62,fuera del ciclo
local,Local,8,8,33,2.23,1.05,3.84,1.38,76.50,0.00,False,-0.39,fuera del ciclo
